# Reproducir las dinámicas de Excel desde `base`
### KinAnalytics · Paletización

**Fuente única:** `Base 2.xlsb` → hoja **`base`**.
Con ella reproducimos las tablas dinámicas de las hojas **`Planilha3`** (imagen 1) y
**`Din`** (imagen 2). Los valores de esas hojas se usan solo como **QA** (comparación).

**Dos clasificaciones (verificadas contra base):**

| columna | regla | acierto |
|---|---|---|
| `valida_oport_palete` (4 cat.) | Picking `pl<0.59` · Palete `floor(pp)≥1 o resto≥0.80` · **Meio `resto≈0.5`** · resto Lastro | **99.94%** filas |
| `oportunidade` (3 cat.) | **`Datas≥3`** y luego `pp≥0.99`→Palete / `pl≥1`→Lastro / resto Sem | **100%** |

donde `pl=%Lastro`, `pp=%Palete`, `resto = pp − floor(pp)`.

## Paso 1 · Leer la fuente y calcular todo

In [1]:
from pathlib import Path
import numpy as np, pandas as pd

XLSB = Path.cwd() / "Base 2.xlsb"
df = pd.read_excel(XLSB, sheet_name="base", engine="pyxlsb")
df.columns = [str(c).strip() for c in df.columns]

# --- derivadas (100% vs base) ---
vol  = df["VOL_CF"].astype(float)
df["pl"] = np.where(df["*Soma Lastro"]!=0, vol/df["*Soma Lastro"], 0.0)   # % Lastro
df["pp"] = np.where(df["* Paletização"]!=0, vol/df["* Paletização"], 0.0) # % Palete
resto = df["pp"] - np.floor(df["pp"])

# --- col22: Valida oport. Palete  (4 categorias) ---
c = np.full(len(df), "Oportunidade Lastro", dtype=object)
c = np.where((np.floor(df["pp"])>=1) | (resto>=0.80), "Oportunidade Palete", c)
c = np.where(np.abs(resto-0.5)<0.005, "Oportunidade Meio Palete", c)     # medio palete exacto
c = np.where(df["pl"]<0.59, "Picking caixas fracionadas", c)
df["valida"] = c

# --- col18: Oportunidade  (3 categorias, gate Datas>=3) ---
gate = df["*Datas Entrega"].astype(float) >= 3
df["oport"] = np.select(
    [~gate, df["pp"]>=0.99, df["pl"]>=1],
    ["Sem Oportunidade","Oportunidade Palete","Oportunidade Lastro"], default="Sem Oportunidade")

d1 = df[df["Dupli. Matricula"]==1].copy()   # clientes distintos (marca del Excel)
print(f"{len(df):,} filas · {len(d1):,} con Dupli.Matricula==1".replace(",", "."))

190.227 filas · 1.668 con Dupli.Matricula==1


In [2]:
# ordenes y helpers de presentacion
ORD4 = ["Oportunidade Lastro","Oportunidade Palete","Oportunidade Meio Palete","Picking caixas fracionadas"]
ORD3 = ["Oportunidade Lastro","Oportunidade Palete","Sem Oportunidade"]
def total_row(t):
    t = t.copy(); t.loc["Total Geral"] = t.sum(); return t
def heat(styler, fmt):
    return styler.background_gradient(cmap="RdYlGn", axis=None).format(fmt)

---
# IMAGEN 1 · hoja `Planilha3`  (clasificación `valida_oport_palete`)

### 1a · Clientes por oportunidad  (conteo de `Dupli.Matricula`)  + %

In [3]:
p3a = d1["valida"].value_counts().reindex(ORD4).to_frame("Dupli. Matricula")
p3a["%"] = p3a["Dupli. Matricula"]/p3a["Dupli. Matricula"].sum()
p3a = total_row(p3a); p3a["%"] = p3a["Dupli. Matricula"]/ (p3a.loc["Total Geral","Dupli. Matricula"])
p3a.style.format({"Dupli. Matricula":"{:,.0f}", "%":"{:.0%}"})

,Dupli. Matricula,%
valida,,
Oportunidade Lastro,777,47%
Oportunidade Palete,819,49%
Oportunidade Meio Palete,70,4%
Picking caixas fracionadas,2,0%
Total Geral,"1,668",100%


### 1b · Volumen (`VOL_CF`) por oportunidad + %  &nbsp; *(Excel: Lastro 2.519.095 · Palete 1.928.940 · Picking 944.275 · Meio 58.835)*

In [4]:
p3b = df.groupby("valida")["VOL_CF"].sum().reindex(ORD4).to_frame("VOL_CF")
p3b["%"] = p3b["VOL_CF"]/p3b["VOL_CF"].sum()
p3b = total_row(p3b); p3b["%"] = p3b["VOL_CF"]/p3b.loc["Total Geral","VOL_CF"]
p3b.style.format({"VOL_CF":"{:,.0f}", "%":"{:.2%}"})

,VOL_CF,%
valida,,
Oportunidade Lastro,"2,529,527",46.40%
Oportunidade Palete,"1,900,923",34.87%
Oportunidade Meio Palete,"76,240",1.40%
Picking caixas fracionadas,"944,455",17.33%
Total Geral,"5,451,145",100.00%


### 1c · % de armado por REDE (Key account)  — *heatmap*

In [5]:
piv = pd.pivot_table(df, index="Key account", columns="valida", values="VOL_CF",
                     aggfunc="sum", fill_value=0).reindex(columns=ORD4, fill_value=0)
piv = piv.loc[piv.sum(axis=1).sort_values(ascending=False).index]   # ordenar por volumen
p3c = piv.div(piv.sum(axis=1), axis=0)
p3c.loc["Total Geral"] = piv.sum()/piv.sum().sum()
heat(p3c.head(20).style, "{:.1%}")

valida,Oportunidade Lastro,Oportunidade Palete,Oportunidade Meio Palete,Picking caixas fracionadas
Key account,,,,
BH SUPERMERCADOS,54.6%,14.1%,1.2%,30.2%
CARREFOUR HIPER,34.8%,57.8%,0.9%,6.5%
PAO DE ACUCAR,53.6%,25.6%,0.6%,20.3%
VILLEFORT,36.9%,58.0%,1.0%,4.0%
EPA,51.1%,19.3%,0.3%,29.2%
SONDA,56.2%,21.9%,2.6%,19.3%
MART MINAS,55.4%,15.2%,7.1%,22.2%
COVABRA,65.2%,20.1%,0.5%,14.3%
DAKI STORE,12.4%,84.6%,2.1%,0.9%


### 1d · Volumen absoluto por REDE  &nbsp; *(Excel BH: Lastro 680.949 · Palete 177.660 · Meio 13.834 · Picking 377.063)*

In [6]:
p3d = total_row(piv)
p3d.head(20).style.format("{:,.0f}").background_gradient(cmap="RdYlGn", axis=None)

valida,Oportunidade Lastro,Oportunidade Palete,Oportunidade Meio Palete,Picking caixas fracionadas
Key account,,,,
BH SUPERMERCADOS,"681,629","175,990","14,824","377,063"
CARREFOUR HIPER,"377,016","625,574","9,733","70,322"
PAO DE ACUCAR,"377,760","180,232","4,056","142,848"
VILLEFORT,"208,242","327,730","5,822","22,786"
EPA,"263,200","99,577","1,762","150,465"
SONDA,"199,233","77,525","9,066","68,503"
MART MINAS,"116,996","32,130","15,041","46,972"
COVABRA,"91,978","28,329",750,"20,119"
DAKI STORE,"16,018","109,001","2,749","1,115"


### 1e · % de armado por Categoría — *heatmap*

In [7]:
pivc = pd.pivot_table(df, index="Categoria", columns="valida", values="VOL_CF",
                      aggfunc="sum", fill_value=0).reindex(columns=ORD4, fill_value=0)
pivc = pivc.loc[pivc.sum(axis=1).sort_values(ascending=False).index]
p3e = pivc.div(pivc.sum(axis=1), axis=0)
p3e.loc["Total Geral"] = pivc.sum()/pivc.sum().sum()
heat(p3e.style, "{:.1%}")

valida,Oportunidade Lastro,Oportunidade Palete,Oportunidade Meio Palete,Picking caixas fracionadas
Categoria,,,,
REFRIG COLAS,43.9%,48.7%,2.2%,5.2%
REFRIG SABORES,49.8%,25.9%,1.2%,23.1%
AGUA,35.9%,55.0%,0.9%,8.2%
SUCOS,59.5%,6.1%,0.5%,33.9%
ENERGETICO,55.0%,18.5%,0.5%,26.0%
CHAS,39.6%,11.1%,0.2%,49.1%
ISOTONICO,57.1%,5.5%,0.4%,37.0%
CERVEJA,37.2%,26.3%,1.0%,35.5%
BEBIDAS VEGETAIS,35.4%,11.4%,0.4%,52.8%


### 1f · Volumen absoluto por Categoría  &nbsp; *(Excel REFRIG COLAS: Lastro 983.065 · Palete 1.109.096 · Meio 39.777 · Picking 117.641)*

In [8]:
total_row(pivc).style.format("{:,.0f}").background_gradient(cmap="RdYlGn", axis=None)

valida,Oportunidade Lastro,Oportunidade Palete,Oportunidade Meio Palete,Picking caixas fracionadas
Categoria,,,,
REFRIG COLAS,"987,780","1,094,814","49,294","117,641"
REFRIG SABORES,"542,473","281,863","13,496","251,998"
AGUA,"224,997","344,075","5,362","51,561"
SUCOS,"305,359","31,469","2,476","173,801"
ENERGETICO,"245,928","82,526","2,340","116,432"
CHAS,"62,633","17,519",352,"77,511"
ISOTONICO,"82,414","7,913",631,"53,374"
CERVEJA,"44,467","31,429","1,201","42,352"
BEBIDAS VEGETAIS,"28,491","9,203",338,"42,511"


---
# IMAGEN 2 · hoja `Din`

### 2a · `valida` × [clientes (Dupli==1), líneas SKU (todas)]  &nbsp; *(Excel: 1.668 / 190.227)*

In [9]:
d_a = pd.DataFrame({
    "Dupli. Matricula": d1["valida"].value_counts(),
    "SKU_ECC":          df["valida"].value_counts(),
}).reindex(ORD4)
total_row(d_a).style.format("{:,.0f}")

,Dupli. Matricula,SKU_ECC
valida,,
Oportunidade Lastro,777,"50,662"
Oportunidade Palete,819,"6,400"
Oportunidade Meio Palete,70,474
Picking caixas fracionadas,2,"132,691"
Total Geral,"1,668","190,227"


### 2b · `oportunidade` × Canal — clientes distintos  &nbsp; *(Excel total 200/115/77/1.276 = 1.668)*

In [10]:
d2 = pd.pivot_table(d1, index="oport", columns="Canal", values="MATRICULA",
                    aggfunc="count", fill_value=0).reindex(ORD3)
d2 = d2.assign(**{"Total Geral": d2.sum(axis=1)})
d2.loc["Total Geral"] = d2.sum()
d2.style.format("{:,.0f}")

Canal,ATACADISTA,HIPERMERCADO,SUPERMERCADO 20 - 49 CKS,SUPERMERCADO 5 -19 CKS,Total Geral
oport,,,,,
Oportunidade Lastro,41,2,10,455,508
Oportunidade Palete,53,67,19,253,392
Sem Oportunidade,106,46,48,568,768
Total Geral,200,115,77,"1,276","1,668"


### 2b' · lo mismo en % por columna  — *heatmap* (como las barras del Excel)

In [11]:
d2p = pd.pivot_table(d1, index="oport", columns="Canal", values="MATRICULA",
                     aggfunc="count", fill_value=0).reindex(ORD3)
d2p = d2p.div(d2p.sum(axis=0), axis=1)
d2p.style.format("{:.1%}").background_gradient(cmap="RdYlGn", axis=None)

Canal,ATACADISTA,HIPERMERCADO,SUPERMERCADO 20 - 49 CKS,SUPERMERCADO 5 -19 CKS
oport,,,,
Oportunidade Lastro,20.5%,1.7%,13.0%,35.7%
Oportunidade Palete,26.5%,58.3%,24.7%,19.8%
Sem Oportunidade,53.0%,40.0%,62.3%,44.5%


### 2c · SKU_ECC_DESC × `oportunidade` — clientes  &nbsp; *(Excel SKU 56443: 220 / 134 / 205)*

In [12]:
d3 = pd.pivot_table(d1, index="SKU_ECC_DESC", columns="oport", values="MATRICULA",
                    aggfunc="count", fill_value=0).reindex(columns=ORD3, fill_value=0)
d3 = d3.assign(**{"Total Geral": d3.sum(axis=1)}).sort_values("Total Geral", ascending=False)
d3.head(20).style.format("{:,.0f}")

oport,Oportunidade Lastro,Oportunidade Palete,Sem Oportunidade,Total Geral
SKU_ECC_DESC,,,,
CC Sem Açúcar PET 200ml 12U FI MAIN,220,134,205,559
Coca-Cola Reg PET 200ml 12U MAINLINE,79,18,108,205
BIPACK CCO+FTALAR PET2L (4) LMPM SCB 2X2,12,50,83,145
CRYSTAL 500ML COM GÁS 12UN,16,37,47,100
CC Pet 600ml 6 Pack FL,27,18,20,65
CC Sem Açúcar LT 220ml 6U FI MAINLINE,45,2,15,62
CC LT6 350ML PROMO SCB,2,23,16,41
CRYSTAL 500ML SEM GÁS 12UN,6,17,15,38
COCA-COLA ORIGINAL PET 2L 8U FL,1,8,20,29


### 2d · REDE (Key account) × `oportunidade` — clientes — *heatmap*  &nbsp; *(Excel BH: 200/97/305 · EPA 67/61/169 · PAO 162/62/9)*

In [13]:
d4 = pd.pivot_table(d1, index="Key account", columns="oport", values="MATRICULA",
                    aggfunc="count", fill_value=0).reindex(columns=ORD3, fill_value=0)
d4 = d4.assign(**{"Total Geral": d4.sum(axis=1)}).sort_values("Total Geral", ascending=False)
d4.head(20).style.format("{:,.0f}").background_gradient(cmap="RdYlGn", axis=None, subset=ORD3)

oport,Oportunidade Lastro,Oportunidade Palete,Sem Oportunidade,Total Geral
Key account,,,,
BH SUPERMERCADOS,200,97,305,602
EPA,67,61,169,297
PAO DE ACUCAR,162,62,9,233
CARREFOUR HIPER,2,67,46,115
MART MINAS,26,14,71,111
SONDA,10,19,48,77
VILLEFORT,1,36,19,56
COVABRA,2,7,38,47
PROENCA,14,10,18,42


---
# QA · comparar contra `Din` y `Planilha3` de `Base 2.xlsb` (lectura en vivo)

Leemos las **hojas reales** `Din` y `Planilha3` de `Base 2.xlsb` (valores ya
calculados por el Excel) y las comparamos, celda a celda, contra lo que
reconstruimos desde `base`.

In [14]:
from pyxlsb import open_workbook
def leer_hoja(name):
    g = {}
    with open_workbook(XLSB) as wb:
        with wb.get_sheet(name) as sh:
            for i, row in enumerate(sh.rows(), start=1):
                for cc in row:
                    g[(i, cc.c)] = cc.v
    return g
DIN = leer_hoja("Din")          # imagen 2
PL3 = leer_hoja("Planilha3")    # imagen 1
print("Hojas de QA leidas de Base 2.xlsb: Din, Planilha3")

Hojas de QA leidas de Base 2.xlsb: Din, Planilha3


### QA-1 · `Din` B4:C7 — conteos por `valida` (clientes y líneas SKU)

In [15]:
din_cli = {DIN[(r,0)]: DIN[(r,1)] for r in range(4,8)}   # Dupli.Matricula
din_sku = {DIN[(r,0)]: DIN[(r,2)] for r in range(4,8)}   # SKU_ECC
qa1 = pd.DataFrame({
    "clientes_Din":  [din_cli[k]              for k in ORD4],
    "clientes_calc": [int((d1['valida']==k).sum()) for k in ORD4],
    "SKU_Din":       [din_sku[k]              for k in ORD4],
    "SKU_calc":      [int((df['valida']==k).sum()) for k in ORD4],
}, index=ORD4)
qa1["dif_cli"] = qa1["clientes_calc"] - qa1["clientes_Din"]
qa1.style.format("{:,.0f}")

,clientes_Din,clientes_calc,SKU_Din,SKU_calc,dif_cli
Oportunidade Lastro,773,777,"50,588","50,662",4
Oportunidade Palete,830,819,"6,521","6,400",-11
Oportunidade Meio Palete,63,70,430,474,7
Picking caixas fracionadas,2,2,"132,688","132,691",0


### QA-2 · `Planilha3` — volumen `VOL_CF` por `valida`  *(la base de los heatmaps 1c–1f)*

In [16]:
pl3_vol = {PL3[(r,0)]: PL3[(r,1)] for r in range(11,15)}
now_vol = df.groupby("valida")["VOL_CF"].sum()
qa2 = pd.DataFrame({"VOL_Planilha3":[pl3_vol[k] for k in ORD4],
                    "VOL_calc":[now_vol[k] for k in ORD4]}, index=ORD4)
qa2["dif_%"] = (qa2["VOL_calc"]-qa2["VOL_Planilha3"])/qa2["VOL_Planilha3"]*100
print("Total  Planilha3 = {:,.0f}   calc = {:,.0f}".format(PL3[(15,1)], now_vol.sum()).replace(",", "."))
qa2.style.format({"VOL_Planilha3":"{:,.0f}","VOL_calc":"{:,.0f}","dif_%":"{:+.2f}%"})

Total  Planilha3 = 5.451.145   calc = 5.451.145


,VOL_Planilha3,VOL_calc,dif_%
Oportunidade Lastro,"2,519,095","2,529,527",+0.41%
Oportunidade Palete,"1,928,940","1,900,923",-1.45%
Oportunidade Meio Palete,"58,835","76,240",+29.58%
Picking caixas fracionadas,"944,275","944,455",+0.02%


### QA-3 · `Din` — `oportunidade` × Canal  (la diferencia debe ser **0** = exacto)

In [17]:
canais = [DIN[(10,c)] for c in range(1,5)]
ref_d2 = pd.DataFrame([[DIN[(r,c)] for c in range(1,5)] for r in range(11,14)],
                      index=[DIN[(r,0)] for r in range(11,14)], columns=canais)
mine_d2 = (pd.pivot_table(d1, index="oport", columns="Canal", values="MATRICULA",
                          aggfunc="count", fill_value=0)
             .reindex(index=ref_d2.index, columns=canais))
dif = (mine_d2 - ref_d2)
print("Referencia (hoja Din):"); print(ref_d2.astype(int))
print("\nDiferencia calc - Din  (0 en todas = reproduccion EXACTA):")
print(dif.astype(int))
print("\nSuma de |diferencias| =", int(dif.abs().values.sum()))

Referencia (hoja Din):
                     ATACADISTA  HIPERMERCADO  SUPERMERCADO 20 - 49 CKS  \
Oportunidade Lastro          41             2                        10   
Oportunidade Palete          53            67                        19   
Sem Oportunidade            106            46                        48   

                     SUPERMERCADO 5 -19 CKS  
Oportunidade Lastro                     455  
Oportunidade Palete                     253  
Sem Oportunidade                        568  

Diferencia calc - Din  (0 en todas = reproduccion EXACTA):
Canal                ATACADISTA  HIPERMERCADO  SUPERMERCADO 20 - 49 CKS  \
Oportunidade Lastro           0             0                         0   
Oportunidade Palete           0             0                         0   
Sem Oportunidade              0             0                         0   

Canal                SUPERMERCADO 5 -19 CKS  
Oportunidade Lastro                       0  
Oportunidade Palete                    

### Conclusión del QA

- **`Din`** (imagen 2, columna `oportunidade`/col18): reproducción **exacta** — la
  matriz de diferencias es **todo ceros** (QA-3), y clientes = 508/392/768.
- **`Planilha3`** (imagen 1, columna `valida`/col22): volumen total **idéntico**
  (5.451.145); por categoría dentro de ~1% (Meio Palete es el ~1% del volumen).
- Conteos por `valida` (QA-1): coinciden salvo unos pocos casos frontera de Meio Palete.

Todo se construyó **solo desde `base`**; `Din` y `Planilha3` se usaron únicamente
para comparar.